In [1]:
# load dataset
from datasets import load_dataset
from sae_lens import SAE
from transformer_lens import HookedTransformer
import torch

# load openwebtext, streaming
dataset = load_dataset("openwebtext", split="train", streaming=True)
model = HookedTransformer.from_pretrained("gpt2", device="mps")

gpt_2_sae, cfg, _ = SAE.from_pretrained(
    release="kvsudarsh/caia-sae-init",
    sae_id="blocks.6.hook_resid_post",
    device="mps"
)

# we have to run inference on a whole bunch of tokens
# for each token, we need to get the SAE encoded features
# then, we look at which features fire here
# we then store that the token caused the feature to fire

# Create dictionaries to store feature-token mappings
feature_to_tokens = {i: [] for i in range(cfg["d_sae"])}
hook_point = cfg["hook_name"]

# Initialize a counter for number of samples processed
n_samples = 0
max_samples = 1000  # Adjust this number based on computational resources

# Iterate through dataset
for text_sample in dataset:
    # Skip if we've processed enough samples
    if n_samples >= max_samples:
        break
        
    # Get tokens from the text
    tokens = model.to_tokens(text_sample['text'])
    
    # Get model activations at the SAE layer
    _, cache = model.run_with_cache(tokens, return_type=None)
    activations = cache["blocks.6.hook_resid_post"]
    
    # Get SAE features
    features = gpt_2_sae.encode(activations)

    
    # for each token, we need to get the features that fire;
    # features have the same shape as the tokens
    # for each feature that fire, we need to append (n_samples, token_idx) to the feature_to_tokens dictionary
    for token_idx in range(tokens.shape[1]):
        token_features = features[0, token_idx]
        # Move to CPU before processing to free MPS memory
        token_features = token_features.cpu()
        fired_features = (token_features > 0).nonzero().squeeze(1).tolist()
        for feature in fired_features:
            feature_to_tokens[feature].append((n_samples, token_idx))
    
    n_samples += 1
    if n_samples % 10 == 0:
        print(f"Processed {n_samples} samples")



/Users/sudarshanagopalkunnavakkam/Documents/Caltech/feature-splitting/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded pretrained model gpt2 into HookedTransformer


/Users/sudarshanagopalkunnavakkam/Documents/Caltech/feature-splitting/venv/lib/python3.11/site-packages/sae_lens/sae.py:145: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(
/Users/sudarshanagopalkunnavakkam/Documents/Caltech/feature-splitting/venv/lib/python3.11/site-packages/sae_lens/sae.py:635: UserWarning: norm_scaling_factor not found for kvsudarsh/caia-sae-init and blocks.6.hook_resid_post, but normalize_activations is 'expected_average_only_in'. Skipping normalization folding.
  warnings.warn(


Processed 10 samples
Processed 20 samples
Processed 30 samples
Processed 40 samples
Processed 50 samples
Processed 60 samples
Processed 70 samples
Processed 80 samples
Processed 90 samples
Processed 100 samples
Processed 110 samples
Processed 120 samples
Processed 130 samples
Processed 140 samples
Processed 150 samples
Processed 160 samples
Processed 170 samples
Processed 180 samples
Processed 190 samples
Processed 200 samples
Processed 210 samples
Processed 220 samples
Processed 230 samples
Processed 240 samples
Processed 250 samples
Processed 260 samples
Processed 270 samples
Processed 280 samples
Processed 290 samples
Processed 300 samples
Processed 310 samples
Processed 320 samples
Processed 330 samples
Processed 340 samples
Processed 350 samples
Processed 360 samples
Processed 370 samples
Processed 380 samples
Processed 390 samples
Processed 400 samples
Processed 410 samples
Processed 420 samples
Processed 430 samples
Processed 440 samples
Processed 450 samples
Processed 460 sampl

KeyboardInterrupt: 